# 🔍 Basic Search Agent with AgentsIQ

This notebook demonstrates how to create a basic search agent using AgentsIQ's intelligent model routing.

## Features:
- Intelligent model selection for different search tasks
- Cost-optimized routing
- Real-time performance monitoring
- Easy-to-use agent framework

## 🚀 Google Colab Ready
This notebook is optimized for Google Colab. Simply run the cells in order and replace the API keys with your actual keys.


In [ ]:
# Install AgentsIQ in Google Colab
%pip install agentsiq

# Set up API keys (replace with your actual keys)
import os
os.environ['OPENAI_API_KEY'] = 'your_openai_key_here'
os.environ['ANTHROPIC_API_KEY'] = 'your_anthropic_key_here'
os.environ['GOOGLE_API_KEY'] = 'your_google_key_here'

print("✅ AgentsIQ installed and configured!")


In [ ]:
import os
import json
from datetime import datetime
from agentsiq.router import ModelRouter
from agentsiq.agent import Agent
from agentsiq.collab import Collab
from agentsiq.decision_store import latest_decisions

print("✅ AgentsIQ imported successfully!")
print(f"📅 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Initialize the Intelligent Router


In [ ]:
# Initialize the router with intelligent model selection
router = ModelRouter()

print("🧠 Available Models:")
for model, profile in router.profiles.items():
    print(f"  {model}: Cost=${profile['cost']:.3f}, Latency={profile['latency']:.2f}s, Quality={profile['quality']:.2f}")

print(f"\n⚙️ Routing Strategy: {router.strategy}")
print(f"⚖️ Weights: Cost={router.weights['cost']:.1%}, Latency={router.weights['latency']:.1%}, Quality={router.weights['quality']:.1%}")


## 2. Create Search Agent


In [ ]:
# Create a search agent with specific capabilities
search_agent = Agent(
    name="WebSearcher",
    description="Searches the web for information and provides comprehensive answers",
    preferred_model="openai:gpt-4o-mini",  # Preferred model (will be overridden by intelligent routing)
    tools=["web_search", "summarize", "analyze"]
)

print(f"🤖 Agent Created: {search_agent.name}")
print(f"📝 Description: {search_agent.description}")
print(f"🔧 Tools: {search_agent.tools}")
print(f"🎯 Preferred Model: {search_agent.preferred}")


## 3. Define Search Tools


In [ ]:
def web_search(query: str) -> str:
    """Simulate web search functionality"""
    # In a real implementation, you would integrate with search APIs
    search_results = {
        "query": query,
        "results": [
            {
                "title": f"Search Result 1 for: {query}",
                "url": "https://example1.com",
                "snippet": f"This is a comprehensive article about {query} with detailed information."
            },
            {
                "title": f"Search Result 2 for: {query}",
                "url": "https://example2.com",
                "snippet": f"Another detailed resource covering {query} from a different perspective."
            },
            {
                "title": f"Search Result 3 for: {query}",
                "url": "https://example3.com",
                "snippet": f"Expert analysis and insights on {query} with practical examples."
            }
        ]
    }
    return json.dumps(search_results, indent=2)

def summarize_content(content: str) -> str:
    """Summarize search results"""
    return f"📋 Summary: {content[:200]}..."

def analyze_trends(data: str) -> str:
    """Analyze trends in search results"""
    return f"📊 Analysis: Based on the search results, here are the key trends and insights..."

# Define tools dictionary
search_tools = {
    "web_search": web_search,
    "summarize": summarize_content,
    "analyze": analyze_trends
}

print("🔧 Search Tools Defined:")
for tool_name, tool_func in search_tools.items():
    print(f"  {tool_name}: {tool_func.__doc__}")


## 4. Test Basic Search Functionality


In [ ]:
# Test the search agent with a simple query
query = "What are the latest trends in artificial intelligence?"

print(f"🔍 Search Query: {query}")
print("\n" + "="*60)

# Let the agent process the query
result = search_agent.act(query, router=router, tools=search_tools)

print(f"🤖 Agent Response:")
print(f"📝 {result['response']}")
print(f"\n🎯 Model Used: {result['model']}")
print(f"⏱️ Processing Time: {result.get('latency', 'N/A')}s")
print(f"💰 Estimated Cost: ${result.get('cost', 'N/A')}")


## 5. Advanced Search with Multiple Queries


In [ ]:
# Test with different types of search queries
search_queries = [
    "How to implement machine learning in Python?",  # Code-related
    "Summarize the benefits of renewable energy",      # Summary task
    "What is the capital of France?",                 # Simple fact
    "Explain quantum computing in simple terms"        # Educational
]

print("🔍 Testing Multiple Search Queries\n")
print("="*80)

for i, query in enumerate(search_queries, 1):
    print(f"\n📝 Query {i}: {query}")
    print("-" * 50)
    
    # Process query
    result = search_agent.act(query, router=router, tools=search_tools)
    
    print(f"🤖 Response: {result['response'][:200]}...")
    print(f"🎯 Model: {result['model']}")
    print(f"⚡ Latency: {result.get('latency', 'N/A')}s")
    print(f"💰 Cost: ${result.get('cost', 'N/A')}")


## 6. View Performance Dashboard


In [ ]:
# View recent decisions and performance metrics
from agentsiq.decision_store import latest_decisions

# Get recent decisions
decisions = latest_decisions(10)
print("📊 Recent Model Selection Decisions:")
print("=" * 60)

for i, decision in enumerate(decisions, 1):
    print(f"\n{i}. Task: {decision.get('task', 'N/A')[:50]}...")
    print(f"   Model Used: {decision.get('chosen', 'N/A')}")
    print(f"   Strategy: {decision.get('strategy', 'N/A')}")
    print(f"   Cost: ${decision.get('est_cost_chosen', 0):.4f}")
    print(f"   Latency: {decision.get('latency', 0):.2f}s")
    print(f"   Quality: {decision.get('quality', 0):.2f}")

# Calculate summary statistics
if decisions:
    total_cost = sum(d.get('est_cost_chosen', 0) for d in decisions)
    avg_latency = sum(d.get('latency', 0) for d in decisions) / len(decisions)
    avg_quality = sum(d.get('quality', 0) for d in decisions) / len(decisions)
    
    print(f"\n📈 Summary Statistics:")
    print(f"   Total Cost: ${total_cost:.4f}")
    print(f"   Average Latency: {avg_latency:.2f}s")
    print(f"   Average Quality: {avg_quality:.3f}")
    print(f"   Total Decisions: {len(decisions)}")


In [ ]:
# Create a simple dashboard visualization
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data for visualization
if decisions:
    df = pd.DataFrame(decisions)
    
    # Create subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Model Usage Count
    model_counts = df['chosen'].value_counts()
    ax1.bar(model_counts.index, model_counts.values, color='skyblue')
    ax1.set_title('Model Usage Distribution')
    ax1.set_xlabel('Models')
    ax1.set_ylabel('Usage Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Cost Analysis
    cost_data = df.groupby('chosen')['est_cost_chosen'].sum().sort_values(ascending=False)
    ax2.bar(cost_data.index, cost_data.values, color='lightcoral')
    ax2.set_title('Total Cost by Model')
    ax2.set_xlabel('Models')
    ax2.set_ylabel('Total Cost ($)')
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. Latency Distribution
    ax3.hist(df['latency'], bins=10, color='lightgreen', alpha=0.7)
    ax3.set_title('Latency Distribution')
    ax3.set_xlabel('Latency (seconds)')
    ax3.set_ylabel('Frequency')
    
    # 4. Quality vs Cost Scatter
    ax4.scatter(df['est_cost_chosen'], df['quality'], alpha=0.7, s=100)
    ax4.set_title('Quality vs Cost Trade-off')
    ax4.set_xlabel('Cost ($)')
    ax4.set_ylabel('Quality Score')
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Dashboard visualization complete!")
else:
    print("⚠️ No decision data available. Run some queries first to see the dashboard.")
